# TESTING AFE

## BMI DATASET

Import the libraries

In [5]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
import os
import sys
project_root = os.path.abspath("")   # notebooks → project root
if project_root not in sys.path:
    sys.path.append(project_root)
from HOFEES.workflow import AutoFEWorkflow
from sklearn.model_selection import train_test_split
import itertools
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold
import optuna

c:\Users\25644574\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Break Train and Test

In [2]:
# -------------------------------------------
# 1) Create synthetic dataset
# -------------------------------------------
np.random.seed(42)
n = 100

height = np.random.normal(1.7, 0.1, n)     # meters
weight = np.random.normal(70, 10, n)       # kg
bmi = weight / (height ** 2)               # BMI formula

# Three noise variables
noise1 = np.random.normal(0, 1, n)
noise2 = np.random.uniform(-1, 1, n)
noise3 = np.random.normal(5, 2, n)

df = pd.DataFrame({
    "height": height,
    "weight": weight,
    "noise1": noise1,
    "noise2": noise2,
    "noise3": noise3,
    "bmi": bmi
})

X = df.drop(columns=["bmi"])
y = df["bmi"]

# -------------------------------------------
# 2) Split into train/test
# -------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

Run HPO

In [4]:
def objective(trial):
    
    # Suggest hyperparameters
    n_int = trial.suggest_int("n_interaction_levels", 1, 4)
    score_bin = trial.suggest_float("score_binary", 0.0, 1.0)
    score_eps = trial.suggest_float("score_eps", 0.0, 1.0)
    dep_th = trial.suggest_float("dep_threshold", 0.0, 0.7)

    kf = KFold(n_splits=3, shuffle=True, random_state=42)
    rmses = []

    for train_idx, val_idx in kf.split(X_train):

        X_subtr = X_train.iloc[train_idx]
        y_subtr = y_train.iloc[train_idx]

        X_val = X_train.iloc[val_idx]
        y_val = y_train.iloc[val_idx]

        # AutoFE
        wf = AutoFEWorkflow(
            task="regression",
            model=RandomForestRegressor(random_state=42),
            metric="neg_root_mean_squared_error",
            iterations=None,
            k_step=1,
            score_binary=score_bin,
            score_eps=score_eps,
            dep_threshold=dep_th,
            n_interaction_levels=n_int,
            random_state=42,
            model_aware_fs_mode='each_stage',
            encode_categoricals=False
        )

        wf.run(X_subtr, y_subtr)

        X_subtr_trans = wf.transform_selected(X_subtr)
        X_val_trans = wf.transform_selected(X_val)

        model = RandomForestRegressor(random_state=42)
        model.fit(X_subtr_trans, y_subtr)

        preds = model.predict(X_val_trans)

        rmse = np.sqrt(mean_squared_error(y_val, preds))
        rmses.append(rmse)

    return np.mean(rmses)

In [6]:
sampler = optuna.samplers.TPESampler(seed=42)

study = optuna.create_study(
    direction="minimize",
    sampler=sampler
)

study.optimize(objective, n_trials=50)

print("Best RMSE:", study.best_value)
print("Best Params:", study.best_params)

[I 2026-06-29 13:37:21,888] A new study created in memory with name: no-name-1722f5e6-f396-4cdd-9fdc-2f75e75e66df
[I 2026-06-29 13:37:24,359] Trial 0 finished with value: 1.456191734633521 and parameters: {'n_interaction_levels': 2, 'score_binary': 0.9507143064099162, 'score_eps': 0.7319939418114051, 'dep_threshold': 0.4190609389379256}. Best is trial 0 with value: 1.456191734633521.
[I 2026-06-29 13:37:26,003] Trial 1 finished with value: 1.1784576164107008 and parameters: {'n_interaction_levels': 1, 'score_binary': 0.15599452033620265, 'score_eps': 0.05808361216819946, 'dep_threshold': 0.6063233020424545}. Best is trial 1 with value: 1.1784576164107008.
[I 2026-06-29 13:37:29,167] Trial 2 finished with value: 1.2376454719947905 and parameters: {'n_interaction_levels': 3, 'score_binary': 0.7080725777960455, 'score_eps': 0.020584494295802447, 'dep_threshold': 0.678936896513396}. Best is trial 1 with value: 1.1784576164107008.
[I 2026-06-29 13:37:34,022] Trial 3 finished with value: 0.4

Best RMSE: 0.4517823435721477
Best Params: {'n_interaction_levels': 4, 'score_binary': 0.21233911067827616, 'score_eps': 0.18182496720710062, 'dep_threshold': 0.12838315689740368}


In [7]:
best = study.best_params

Assessing the feature

In [9]:
# -------------------------------------------
# 1) Rebuild workflow with best parameters
# -------------------------------------------
wf_best = AutoFEWorkflow(
    task="regression",
    model=RandomForestRegressor(random_state=42),
    metric="neg_root_mean_squared_error",
    iterations=None,
    k_step=1,
    score_binary=best['score_binary'],
    score_eps=0,
    dep_threshold=best['dep_threshold'],
    n_interaction_levels=best['n_interaction_levels'],
    random_state=42,
    model_aware_fs_mode='each_stage',
    encode_categoricals=False
)

# -------------------------------------------
# 2) Fit on FULL training data
# -------------------------------------------
wf_best.run(X_train, y_train)

# -------------------------------------------
# 3) Transform test set
# -------------------------------------------
X_test_trans = wf_best.transform_selected(X_test)

# -------------------------------------------
# 4) Print columns
# -------------------------------------------
print("Transformed Test Columns:")
print(X_test_trans.columns.tolist())

Transformed Test Columns:
['U[identity](U[identity](B[div](U[square](B[mul](U[reciprocal](height),U[identity](weight))),U[identity](U[identity](weight)))))']
